# Day 4 — Feature Engineering, PCA & Ensembles

This notebook extends the RetailMind AI churn workflow. All transformations are fit only on training folds; the held-out test split is used once for final reporting.

In [ ]:
from pathlib import Path
import json
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
CV = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
BASE_DIR = Path.cwd()
if not (BASE_DIR / 'data').exists():
    BASE_DIR = BASE_DIR.parent
DATA_PATH = BASE_DIR / 'data' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
MODEL_DIR = BASE_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)
assert DATA_PATH.exists(), f'Dataset not found: {DATA_PATH}'
print(f'Project directory: {BASE_DIR}')
print(f'Dataset: {DATA_PATH}')

In [ ]:
class TelcoFeatureEngineer(BaseEstimator, TransformerMixin):
    """Deterministic, row-level Telco features; it learns no dataset statistics."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        frame = X.copy()
        frame = frame.drop(columns=['customerID', 'Churn'], errors='ignore')
        frame['TotalCharges'] = pd.to_numeric(frame['TotalCharges'], errors='coerce')
        tenure = pd.to_numeric(frame['tenure'], errors='coerce').fillna(0)
        monthly = pd.to_numeric(frame['MonthlyCharges'], errors='coerce')
        total = frame['TotalCharges']
        frame['tenure_bucket'] = pd.cut(tenure, [-1, 6, 24, 48, np.inf],
                                        labels=['new', 'short_term', 'medium_term', 'long_term']).astype(str)
        frame['average_monthly_charge_by_tenure'] = (total / tenure.replace(0, np.nan)).fillna(monthly)
        frame['monthly_to_total_charge_ratio'] = monthly / total.replace(0, np.nan)
        frame['charge_tenure_interaction'] = monthly * tenure
        frame['charge_vs_average_gap'] = monthly - frame['average_monthly_charge_by_tenure']
        frame['is_month_to_month'] = (frame['Contract'] == 'Month-to-month').astype(int)
        frame['is_long_term_contract'] = frame['Contract'].isin(['One year', 'Two year']).astype(int)
        frame['has_fiber_optic'] = (frame['InternetService'] == 'Fiber optic').astype(int)
        frame['uses_electronic_check'] = (frame['PaymentMethod'] == 'Electronic check').astype(int)
        frame['uses_auto_payment'] = frame['PaymentMethod'].str.contains('automatic', case=False, na=False).astype(int)
        frame['paperless_electronic_check'] = ((frame['PaperlessBilling'] == 'Yes') &
                                               (frame['PaymentMethod'] == 'Electronic check')).astype(int)
        add_on_columns = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
                          'StreamingTV', 'StreamingMovies']
        frame['service_count'] = frame[add_on_columns].eq('Yes').sum(axis=1) + frame['PhoneService'].eq('Yes').astype(int)
        frame['has_security_or_support'] = ((frame['OnlineSecurity'] == 'Yes') |
                                            (frame['TechSupport'] == 'Yes')).astype(int)
        frame['loyalty_score'] = (tenure / 12) + 2 * frame['is_long_term_contract'] + frame['Partner'].eq('Yes').astype(int)
        return frame

def make_preprocessor(numeric_columns, categorical_columns):
    return ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_columns),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                  ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_columns),
    ], sparse_threshold=0, verbose_feature_names_out=False)

df = pd.read_csv(DATA_PATH)
X = df.drop(columns='Churn')
y = df['Churn'].map({'Yes': 1, 'No': 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
engineered_train = TelcoFeatureEngineer().fit_transform(X_train)
numeric_columns = engineered_train.select_dtypes(include=['number']).columns.tolist()
categorical_columns = engineered_train.select_dtypes(exclude=['number']).columns.tolist()
print(f'Train/test rows: {len(X_train)}/{len(X_test)}; engineered features before encoding: {engineered_train.shape[1]}')
print('Churn rate:', round(y.mean(), 4))

In [ ]:
# Existing baseline: its original raw columns and logistic-regression design.
raw_columns = [column for column in X_train.columns if column != 'customerID']
raw_train = TelcoFeatureEngineer().fit_transform(X_train)[raw_columns]
raw_numeric = raw_train.select_dtypes(include=['number']).columns.tolist()
raw_categorical = raw_train.select_dtypes(exclude=['number']).columns.tolist()
# The feature engineer normalizes TotalCharges, while this preprocessor selects only original columns.
raw_baseline = Pipeline([
    ('feature_engineer', TelcoFeatureEngineer()),
    ('preprocessor', make_preprocessor(raw_numeric, raw_categorical)),
    ('classifier', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

def engineered_pipeline(classifier, include_pca=False):
    steps = [('feature_engineer', TelcoFeatureEngineer()),
             ('preprocessor', make_preprocessor(numeric_columns, categorical_columns))]
    if include_pca:
        steps.append(('pca', PCA(n_components=0.95, random_state=RANDOM_STATE)))
    steps.append(('classifier', classifier))
    return Pipeline(steps)

negative_to_positive_ratio = (y_train == 0).sum() / (y_train == 1).sum()
rf_search = GridSearchCV(
    engineered_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced')),
    {'classifier__n_estimators': [250], 'classifier__max_depth': [8, None],
     'classifier__min_samples_leaf': [3], 'classifier__max_features': ['sqrt']},
    scoring='roc_auc', cv=CV, n_jobs=1, refit=True)
xgb_search = GridSearchCV(
    engineered_pipeline(XGBClassifier(random_state=RANDOM_STATE, n_jobs=1, tree_method='hist',
                                       eval_metric='logloss', scale_pos_weight=negative_to_positive_ratio)),
    {'classifier__n_estimators': [200], 'classifier__max_depth': [3, 4],
     'classifier__learning_rate': [0.05], 'classifier__subsample': [0.8],
     'classifier__colsample_bytree': [0.8]},
    scoring='roc_auc', cv=CV, n_jobs=1, refit=True)

print('Tuning Random Forest...')
rf_search.fit(X_train, y_train)
print('Tuning XGBoost...')
xgb_search.fit(X_train, y_train)
pca_model = engineered_pipeline(LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE), include_pca=True)
raw_baseline.fit(X_train, y_train)
pca_model.fit(X_train, y_train)
models = {'Existing baseline (Logistic Regression)': raw_baseline, 'Random Forest': rf_search.best_estimator_,
          'XGBoost': xgb_search.best_estimator_, 'PCA Logistic Regression': pca_model}
print('RF best parameters:', rf_search.best_params_)
print('XGBoost best parameters:', xgb_search.best_params_)


In [ ]:
# PCA is fitted on the training data only. This full PCA view is solely for redundancy analysis.
analysis_preprocessor = make_preprocessor(numeric_columns, categorical_columns)
encoded_train = analysis_preprocessor.fit_transform(engineered_train)
full_pca = PCA(random_state=RANDOM_STATE).fit(encoded_train)
cumulative_variance = np.cumsum(full_pca.explained_variance_ratio_)
chosen_components = int(np.searchsorted(cumulative_variance, 0.95) + 1)
variance_table = pd.DataFrame({'component': np.arange(1, len(cumulative_variance) + 1),
                               'explained_variance_ratio': full_pca.explained_variance_ratio_,
                               'cumulative_explained_variance': cumulative_variance})
variance_table.to_csv(MODEL_DIR / 'day4_pca_explained_variance.csv', index=False)
plt.figure(figsize=(8, 4))
plt.plot(variance_table['component'], variance_table['cumulative_explained_variance'], marker='o', markersize=2)
plt.axhline(0.95, color='crimson', linestyle='--', label='95% threshold')
plt.axvline(chosen_components, color='grey', linestyle=':', label=f'{chosen_components} components')
plt.xlabel('Principal components'); plt.ylabel('Cumulative explained variance'); plt.legend(); plt.tight_layout()
plt.savefig(MODEL_DIR / 'day4_pca_cumulative_variance.png', dpi=160)
plt.show()
print(f'Encoded feature count: {encoded_train.shape[1]}; components for >=95% variance: {chosen_components}')
print('PCA pipeline selected components:', pca_model.named_steps['pca'].n_components_)


In [ ]:
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
records, reports, matrices = [], {}, {}
for name, model in models.items():
    train_input = X_train
    test_input = X_test
    cv_scores = cross_validate(model, train_input, y_train, scoring=scoring, cv=CV, n_jobs=1)
    model.fit(train_input, y_train)
    prediction = model.predict(test_input)
    probability = model.predict_proba(test_input)[:, 1]
    records.append({'Model': name,
                    **{f'CV {metric.replace('_', ' ').title()}': cv_scores[f'test_{metric}'].mean() for metric in scoring},
                    'Accuracy': accuracy_score(y_test, prediction), 'Precision': precision_score(y_test, prediction, zero_division=0),
                    'Recall': recall_score(y_test, prediction, zero_division=0), 'F1-score': f1_score(y_test, prediction, zero_division=0),
                    'ROC-AUC': roc_auc_score(y_test, probability)})
    reports[name] = classification_report(y_test, prediction, output_dict=True, zero_division=0)
    matrices[name] = confusion_matrix(y_test, prediction).tolist()
comparison_df = pd.DataFrame(records).sort_values(['CV Roc Auc', 'CV F1', 'CV Recall'], ascending=False).reset_index(drop=True)
comparison_df.to_csv(MODEL_DIR / 'day4_model_comparison.csv', index=False)
display(comparison_df.round(4))
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for axis, (name, matrix) in zip(axes.ravel(), matrices.items()):
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axis)
    axis.set_title(name); axis.set_xlabel('Predicted'); axis.set_ylabel('Actual')
plt.tight_layout(); plt.savefig(MODEL_DIR / 'day4_confusion_matrices.png', dpi=160); plt.show()
for name, report in reports.items():
    print(f'\n===== {name} classification report =====')
    print(pd.DataFrame(report).transpose().round(3))


In [ ]:
# Champion selection is based on stratified cross-validation, with ROC-AUC, F1 and recall prioritized in that order.
champion_name = comparison_df.loc[0, 'Model']
champion = models[champion_name]
# Persist only standard sklearn objects so joblib.load works in a fresh process.
# Raw records are first passed through the deterministic feature-engineering cell above.
inference_steps = [(name, step) for name, step in champion.steps if name != 'feature_engineer']
inference_model = Pipeline(inference_steps)
joblib.dump(inference_model, MODEL_DIR / 'day4_churn_champion.joblib')
joblib.dump(inference_model.named_steps['preprocessor'], MODEL_DIR / 'day4_preprocessor.joblib')
joblib.dump(pca_model.named_steps['pca'], MODEL_DIR / 'day4_pca.joblib')
feature_names = inference_model.named_steps['preprocessor'].get_feature_names_out().tolist()
metadata = {'champion': champion_name, 'selection_rule': 'highest 3-fold CV ROC-AUC, then CV F1 and CV recall',
            'dataset': str(DATA_PATH.name), 'random_state': RANDOM_STATE,
            'inference_input': 'Use TelcoFeatureEngineer in this notebook to transform raw Telco records before calling the persisted sklearn pipeline.',
            'engineered_features': ['tenure_bucket', 'average_monthly_charge_by_tenure', 'monthly_to_total_charge_ratio',
              'charge_tenure_interaction', 'charge_vs_average_gap', 'is_month_to_month', 'is_long_term_contract',
              'has_fiber_optic', 'uses_electronic_check', 'uses_auto_payment', 'paperless_electronic_check',
              'service_count', 'has_security_or_support', 'loyalty_score'],
            'encoded_feature_names': feature_names, 'pca_components_for_95_percent_variance': chosen_components,
            'random_forest_best_params': rf_search.best_params_, 'xgboost_best_params': xgb_search.best_params_,
            'test_classification_reports': reports, 'test_confusion_matrices': matrices}
with open(MODEL_DIR / 'day4_metadata.json', 'w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)
reloaded = joblib.load(MODEL_DIR / 'day4_churn_champion.joblib')
sample = TelcoFeatureEngineer().transform(X_test.iloc[[0]])
sample_prediction = int(reloaded.predict(sample)[0])
sample_probability = float(reloaded.predict_proba(sample)[0, 1])
print(f'Champion: {champion_name}')
print(f'Persistence verified — sample prediction: {sample_prediction}, probability: {sample_probability:.4f}')